# NB_hospital_daily_summary
Builds `<catalog>.gold.hospital_daily_summary` per the STTM (`sttm_silver_to_gold.xlsx`, sheet *hospital_daily_summary*).

* **Grain:** one row per hospital per calendar day (days with at least one lab test)
* **Driving table:** `<catalog>.silver.lab_results` grouped by `hospital_id`, `test_date`
* **Joins (LEFT):** `silver.hospitals` on `hospital_id`; calendar on `date_key = yyyyMMdd(test_date)`
* **Window:** `LAG(total_tests)` partitioned by `hospital_id`, ordered by `summary_date`
* **Load:** FULL - `CREATE OR REPLACE TABLE ... AS SELECT`

Note: the calendar table exists in silver as `calender` (STTM spells it `calendar`).

In [0]:
dbutils.widgets.text("catalog", "adb_caren")
catalog = dbutils.widgets.get("catalog").strip()

source_tables = ["lab_results", "hospitals", "calender"]
missing = [t for t in source_tables if not spark.catalog.tableExists(f"{catalog}.silver.{t}")]
if missing:
    raise ValueError(f"Missing source tables in {catalog}.silver: {missing}")

spark.sql("CREATE SCHEMA IF NOT EXISTS IDENTIFIER(:catalog || '.gold')", args={"catalog": catalog})

In [0]:
hospital_daily_summary_sql = """
CREATE OR REPLACE TABLE IDENTIFIER(:catalog || '.gold.hospital_daily_summary') AS
WITH daily AS (
  SELECT hospital_id,
         test_date                            AS summary_date,
         COUNT(*)                             AS total_tests,
         COUNT(DISTINCT patient_id)           AS unique_patients_tested,
         COUNT(DISTINCT doctor_id)            AS active_doctor_count,
         COUNT_IF(result_status = 'Abnormal') AS abnormal_test_count,
         COUNT_IF(result_status = 'Critical') AS critical_test_count
  FROM IDENTIFIER(:catalog || '.silver.lab_results')
  GROUP BY hospital_id, test_date
),
with_lag AS (
  SELECT d.*,
         LAG(total_tests) OVER (PARTITION BY hospital_id ORDER BY summary_date) AS prev_day_total_tests
  FROM daily d
)
SELECT
  CAST(w.hospital_id AS INT)                                                 AS hospital_id,
  h.hospital_name,
  w.summary_date,
  c.day_name,
  c.is_weekend,
  CAST(w.total_tests AS INT)                                                 AS total_tests,
  CAST(w.unique_patients_tested AS INT)                                      AS unique_patients_tested,
  CAST(w.active_doctor_count AS INT)                                         AS active_doctor_count,
  CAST(w.abnormal_test_count AS INT)                                         AS abnormal_test_count,
  CAST(w.critical_test_count AS INT)                                         AS critical_test_count,
  CAST(ROUND(w.abnormal_test_count / w.total_tests * 100, 2) AS DECIMAL(5,2)) AS abnormal_rate_pct,
  CAST(ROUND(w.critical_test_count / w.total_tests * 100, 2) AS DECIMAL(5,2)) AS critical_rate_pct,
  CAST(w.prev_day_total_tests AS INT)                                        AS prev_day_total_tests,
  CAST(ROUND((w.total_tests - w.prev_day_total_tests)
             / NULLIF(w.prev_day_total_tests, 0) * 100, 2) AS DECIMAL(6,2))  AS test_volume_change_pct
FROM with_lag w
LEFT JOIN IDENTIFIER(:catalog || '.silver.hospitals') h
  ON w.hospital_id = h.hospital_id
LEFT JOIN IDENTIFIER(:catalog || '.silver.calender') c
  ON c.date_key = CAST(date_format(w.summary_date, 'yyyyMMdd') AS INT)
"""

spark.sql(hospital_daily_summary_sql, args={"catalog": catalog})
print(f"Rebuilt {catalog}.gold.hospital_daily_summary")

In [0]:
gold_df = spark.table(f"{catalog}.gold.hospital_daily_summary")
gold_count = gold_df.count()
distinct_keys = gold_df.select("hospital_id", "summary_date").distinct().count()
gold_tests = gold_df.agg({"total_tests": "sum"}).collect()[0][0]
source_tests = spark.table(f"{catalog}.silver.lab_results").count()

print(f"gold rows: {gold_count} | tests in gold: {gold_tests} | tests in silver: {source_tests}")
assert distinct_keys == gold_count, "Duplicate (hospital_id, summary_date) found - grain violated"
assert gold_tests == source_tests, "Total tests do not reconcile with silver.lab_results"

display(gold_df.orderBy("hospital_id", "summary_date").limit(20))

In [0]:
# Final summary
target_table = f"{catalog}.gold.hospital_daily_summary"
load_type_normalized = "FULL"

result_df = spark.read.table(target_table)
result_count = result_df.count()
print(f"\n{'='*60}")
print(f"✅ {load_type_normalized} load completed for {target_table}")
print(f"📊 Total rows in target table after load: {result_count}")
print(f"{'='*60}")

dbutils.notebook.exit(str(result_count))